In [ ]:
"""
Data-Efficient DFL: Shortest Path (Pool-Based Version)
=======================================================

Methods:
  supervise:   Labels the first budget samples in pool order, no selection.
  margin:      Each step selects the sample with the smallest margin
               (closest to the SPO decision boundary).

  uncertainty: Each step selects the sample farthest from the labeled set with largest variance

  flip:        Each step selects the sample with the highest flip probability.
  regret:      Each step selects the sample with the highest expected regret.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import time
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# Part 1: Grid
# ============================================================

def build_grid(grid_size):
    n = grid_size
    n_h = n * (n - 1); n_v = (n - 1) * n; d = n_h + n_v
    total_steps = 2 * (n - 1)
    all_paths = []
    for right_pos in combinations(range(total_steps), n - 1):
        right_set = set(right_pos)
        edges = []; ci = cj = 0
        for step in range(total_steps):
            if step in right_set:
                edges.append(ci * (n-1) + cj); cj += 1
            else:
                edges.append(n_h + cj * (n-1) + ci); ci += 1
        all_paths.append(np.array(edges))
    return d, len(all_paths), all_paths


def solve_shortest_path(cost, gs):
    n = gs; d = len(cost); n_h = n*(n-1)
    h = np.zeros((n, n-1)); v = np.zeros((n-1, n))
    idx = 0
    for i in range(n):
        for j in range(n-1): h[i,j] = cost[idx]; idx += 1
    for j in range(n):
        for i in range(n-1): v[i,j] = cost[idx]; idx += 1
    INF = 1e18
    dist = np.full((n,n), INF); par = np.full((n,n,2), -1, dtype=int)
    dist[0,0] = 0
    for i in range(n):
        for j in range(n):
            if i==0 and j==0: continue
            if j>0 and dist[i,j-1]+h[i,j-1] < dist[i,j]:
                dist[i,j] = dist[i,j-1]+h[i,j-1]; par[i,j] = [i,j-1]
            if i>0 and dist[i-1,j]+v[i-1,j] < dist[i,j]:
                dist[i,j] = dist[i-1,j]+v[i-1,j]; par[i,j] = [i-1,j]
    w = np.zeros(d); ci = cj = n-1
    while not (ci==0 and cj==0):
        pi, pj = par[ci,cj]
        if pi==ci and pj==cj-1: w[ci*(n-1)+(cj-1)] = 1.0
        elif pi==ci-1 and pj==cj: w[n*(n-1)+cj*(n-1)+(ci-1)] = 1.0
        ci, cj = pi, pj
    return w


def solve_sp_torch(cost, gs):
    c = cost.detach().cpu().numpy()
    if c.ndim == 1:
        return torch.FloatTensor(solve_shortest_path(c, gs)).to(cost.device)
    return torch.FloatTensor(
        np.array([solve_shortest_path(ci, gs) for ci in c])
    ).to(cost.device)


def get_opt_path_idx(cost, gs, all_paths):
    w = solve_shortest_path(cost, gs)
    active = set(np.where(w > 0.5)[0])
    for i, p in enumerate(all_paths):
        if set(p) == active: return i
    return -1


# ============================================================
# Part 2: Data Generation
# ============================================================

def generate_B_matrix(p, d, gs, all_paths, margin_threshold=0.1,
                      max_attempts=1000, rng=None):
    if rng is None: rng = np.random.RandomState(42)
    n_paths = len(all_paths)
    thresholds_to_try = [margin_threshold, 0.01]

    for thresh in thresholds_to_try:
        for attempt in range(max_attempts):
            B = rng.binomial(1, 0.5, size=(d, p)).astype(np.float64)
            centers = []; ok = True
            for pi in range(n_paths):
                found = False
                for scale in [0.5, 1.0]:
                    for _ in range(100):
                        mu = rng.randn(p) * scale
                        cost = B @ mu
                        if get_opt_path_idx(cost, gs, all_paths) == pi:
                            w = solve_shortest_path(cost, gs)
                            oc = cost @ w
                            min_gap = np.inf
                            for pj, path in enumerate(all_paths):
                                if pj != pi:
                                    wp = np.zeros(d); wp[path] = 1.0
                                    g = cost @ wp - oc
                                    if g < min_gap: min_gap = g
                            if min_gap > thresh:
                                centers.append(mu); found = True; break
                    if found: break
                if not found: ok = False; break
            if ok:
                if thresh < margin_threshold:
                    print(f"  Note: used margin_threshold={thresh:.4f}")
                return B, centers

    B = rng.binomial(1, 0.5, size=(d, p)).astype(np.float64)
    return B, [rng.randn(p)*1.0 for _ in range(n_paths)]


def generate_data(n, p, d, gs, all_paths, noise_level=0.25, deg=1,
                  seed=42, B=None, centers=None):
    rng = np.random.RandomState(seed)
    n_paths = len(all_paths); sigma_m = 1.0/3.0
    if B is None or centers is None:
        B, centers = generate_B_matrix(p, d, gs, all_paths, rng=rng)
    X = np.zeros((n, p))
    cluster_assign = np.zeros(n, dtype=int)
    for i in range(n):
        j = rng.randint(0, n_paths)
        cluster_assign[i] = j
        X[i] = rng.randn(p) * sigma_m + centers[j]
    lp = X @ B.T / np.sqrt(p)
    base = 1.0 + (1.0 + lp) ** deg
    eps = rng.uniform(1-noise_level, 1+noise_level, size=(n, d)) \
          if noise_level > 0 else np.ones((n, d))
    return X, base * eps, B, centers, cluster_assign


def compute_bayes_risk(X_test, C_test, B, gs, p, deg=1):
    C_oracle = 1.0 + (1.0 + X_test @ B.T / np.sqrt(p)) ** deg
    n = len(X_test); total = 0.0
    for i in range(n):
        wo = solve_shortest_path(C_oracle[i], gs)
        wt = solve_shortest_path(C_test[i], gs)
        total += C_test[i] @ wo - C_test[i] @ wt
    return total / n


# ============================================================
# Part 3: Pool Order Generation
# ============================================================

def make_pool_order(cluster_assign, clustering_strength, rng):
    """
    Generate pool access order using a Markov chain to control clustering degree.

    clustering_strength:
      0.0 = fully random (i.i.d.)
      1.0 = strictly clustered
      intermediate = P(stay in current cluster) = clustering_strength
    """
    n = len(cluster_assign)

    if clustering_strength <= 1e-8:
        return rng.permutation(n)

    n_clusters = int(cluster_assign.max()) + 1
    by_cluster = {j: list(np.where(cluster_assign == j)[0])
                  for j in range(n_clusters)}
    for j in by_cluster:
        rng.shuffle(by_cluster[j])

    if clustering_strength >= 1.0 - 1e-8:
        cluster_order = list(by_cluster.keys())
        rng.shuffle(cluster_order)
        order = []
        for j in cluster_order:
            order.extend(by_cluster[j])
        return np.array(order)

    # Markov chain
    order = []
    cluster_pointers = {j: 0 for j in by_cluster}
    avail = [j for j in by_cluster if len(by_cluster[j]) > 0]
    current = rng.choice(avail)
    while len(order) < n:
        if cluster_pointers[current] < len(by_cluster[current]):
            if rng.rand() < clustering_strength:
                idx = by_cluster[current][cluster_pointers[current]]
                order.append(idx); cluster_pointers[current] += 1
                continue
        avail = [j for j in by_cluster
                 if cluster_pointers[j] < len(by_cluster[j])]
        if not avail: break
        non_cur = [j for j in avail if j != current]
        current = rng.choice(non_cur) if non_cur else avail[0]
        idx = by_cluster[current][cluster_pointers[current]]
        order.append(idx); cluster_pointers[current] += 1

    return np.array(order)


# ============================================================
# Part 4: SPO+ Loss
# ============================================================

class SPOPlusLoss(nn.Module):
    def __init__(self, gs):
        super().__init__(); self.gs = gs

    def forward(self, cp, ct):
        with torch.no_grad():
            wt = solve_sp_torch(ct, self.gs)
            ws = solve_sp_torch(2*cp - ct, self.gs)
        return torch.clamp(
            torch.sum((ct-2*cp)*ws, 1) + torch.sum(2*cp*wt, 1) -
            torch.sum(ct*wt, 1), min=0.0
        ).mean()


# ============================================================
# Part 5: Model
# ============================================================

class LinearModel(nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.layer = nn.Linear(inp, out)

    def forward(self, x): return self.layer(x)

    def get_features(self, x):
        with torch.no_grad(): return x


class MLPModel(nn.Module):
    def __init__(self, inp, hid, out):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Linear(inp, hid), nn.ReLU(),
            nn.Linear(hid, hid), nn.ReLU())
        self.last_layer = nn.Linear(hid, out)

    def forward(self, x): return self.last_layer(self.feat(x))

    def get_features(self, x):
        with torch.no_grad(): return self.feat(x)


def make_model(inp, hid, out, model_type='linear'):
    if model_type == 'linear':
        return LinearModel(inp, out)
    return MLPModel(inp, hid, out)


# ============================================================
# Part 6: Train and Evaluation
# ============================================================

def train_init(model, X, C, gs, device='cpu', epochs=10, lr=0.01):
    model.train(); model.to(device)
    Xt = torch.FloatTensor(X).to(device)
    Ct = torch.FloatTensor(C).to(device)
    crit = SPOPlusLoss(gs); opt = optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); crit(model(Xt), Ct).backward(); opt.step()
    model.eval()


def train_update(model, X, C, gs, device='cpu', steps=3, lr=0.01):
    model.train(); model.to(device)
    Xt = torch.FloatTensor(X).to(device)
    Ct = torch.FloatTensor(C).to(device)
    crit = SPOPlusLoss(gs); opt = optim.SGD(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad(); crit(model(Xt), Ct).backward(); opt.step()
    model.eval()


def train_online_step(model, x_new, c_new, gs, device='cpu', lr=0.05):
    model.train(); model.to(device)
    Xt = torch.FloatTensor(x_new).unsqueeze(0).to(device)
    Ct = torch.FloatTensor(c_new).unsqueeze(0).to(device)
    crit = SPOPlusLoss(gs); opt = optim.SGD(model.parameters(), lr=lr)
    opt.zero_grad(); crit(model(Xt), Ct).backward(); opt.step()
    model.eval()


def evaluate(model, Xt, Ct, gs, device='cpu'):
    model.eval()
    with torch.no_grad():
        cp = model(torch.FloatTensor(Xt).to(device)).cpu().numpy()
    n = len(Xt); reg = 0.0; exact = 0
    for i in range(n):
        wp = solve_shortest_path(cp[i], gs)
        wt = solve_shortest_path(Ct[i], gs)
        reg += Ct[i] @ wp - Ct[i] @ wt
        if np.array_equal(wp, wt): exact += 1
    return {'regret': reg/n, 'path_accuracy': exact/n}


# ============================================================
# Part 7: Last-Layer Laplace (used by flip/regret)
# ============================================================

class LastLayerLaplace:
    def __init__(self, model, prior_prec=1.0, noise_var=1.0):
        self.model = model; self.pp = prior_prec; self.nv = noise_var
        self.Sigma = None

    def fit(self, X, device='cpu'):
        self.model.eval()
        with torch.no_grad():
            phi = self.model.get_features(X).cpu().numpy()
        n, p = phi.shape
        self.Sigma = np.linalg.inv(
            (1.0/self.nv) * (phi.T @ phi) + self.pp * np.eye(p))

    def predict_batch(self, X, device='cpu'):
        self.model.eval()
        with torch.no_grad():
            mus = self.model(X).cpu().numpy()
            phis = self.model.get_features(X).cpu().numpy()
        vars_ = np.sum((phis @ self.Sigma) * phis, axis=1)
        return mus, vars_


# ============================================================
# Part 8: Pool-Based Scoring Functions
# ============================================================

_PATH_VECTORS_CACHE = {}

def get_path_vectors(gs, d, all_paths):
    key = (gs, d)
    if key not in _PATH_VECTORS_CACHE:
        Ws = np.zeros((len(all_paths), d))
        for i, path in enumerate(all_paths):
            Ws[i, path] = 1.0
        _PATH_VECTORS_CACHE[key] = Ws
    return _PATH_VECTORS_CACHE[key]


def compute_margin_single(c_pred, gs, all_paths):
    """nu_S(c) = min_{v_j!=w*(c)} c^T(v_j-w*(c)) / ||v_j-w*(c)||_2"""
    d = len(c_pred)
    Ws = get_path_vectors(gs, d, all_paths)
    costs = Ws @ c_pred
    best_idx = int(np.argmin(costs))
    w_best = Ws[best_idx]; min_val = np.inf
    for j in range(len(all_paths)):
        if j == best_idx: continue
        diff = Ws[j] - w_best
        val = (c_pred @ diff) / max(np.linalg.norm(diff, 2), 1e-12)
        if val < min_val: min_val = val
    return min_val


def compute_flip_prob(mu, sigma, gs, M=30):
    d = len(mu)
    w_hat = solve_shortest_path(mu, gs)
    samples = np.maximum(
        mu[np.newaxis, :] + sigma * np.random.randn(M, d), 1e-6)
    flips = 0; reg = 0.0
    for m in range(M):
        ws = solve_shortest_path(samples[m], gs)
        if not np.array_equal(ws, w_hat): flips += 1
        reg += max(samples[m] @ w_hat - samples[m] @ ws, 0)
    return flips/M, reg/M


def score_pool(model, X_pool, X_train_current,
               unlabeled_idx, method, gs, all_paths,
               laplace=None, device='cpu', M=30):

    X_unlabeled = X_pool[unlabeled_idx]
    n = len(unlabeled_idx)

    if method == 'margin':
        model.eval()
        Xt = torch.FloatTensor(X_unlabeled).to(device)
        with torch.no_grad():
            cp = model(Xt).cpu().numpy()
        margins = np.array([compute_margin_single(c, gs, all_paths)
                             for c in cp])
        return -margins  # negate: smaller margin -> larger score

    elif method == 'uncertainty':
        scores = np.zeros(n)
        for i, idx in enumerate(unlabeled_idx):
            x = X_pool[idx]
            dists = np.linalg.norm(X_train_current - x, axis=1)
            scores[i] = dists.min()
        return scores

    elif method == 'flip':
        model.eval()
        Xt = torch.FloatTensor(X_unlabeled).to(device)
        mus, vars_ = laplace.predict_batch(Xt, device)
        scores = np.zeros(n)
        for i in range(n):
            sigma = np.sqrt(max(vars_[i], 1e-10))
            fp, _ = compute_flip_prob(mus[i], sigma, gs, M)
            scores[i] = fp
        return scores

    elif method == 'regret':
        model.eval()
        Xt = torch.FloatTensor(X_unlabeled).to(device)
        mus, vars_ = laplace.predict_batch(Xt, device)
        scores = np.zeros(n)
        for i in range(n):
            sigma = np.sqrt(max(vars_[i], 1e-10))
            _, ar = compute_flip_prob(mus[i], sigma, gs, M)
            scores[i] = ar
        return scores

    else:
        raise ValueError(f"Unknown method: {method}")


# ============================================================
# Part 9: Visualization
# ============================================================

STYLES = {
    'supervise': {'color': '#FF8C00', 'ls': '-',  'marker': 'o',
                  'label': 'Supervised Learning'},
    'uncertainty': {'color': 'green', 'ls': '-.', 'marker': 's',
                    'label': 'Uncertainty'},
    'margin': {'color': '#1E90FF', 'ls': '-', 'marker': '^',
               'label': 'Margin Based Algorithm'},
    'flip': {'color': 'red', 'ls': '-', 'marker': 'D',
             'label': 'Flip Prob'},
    'regret': {'color': 'darkred', 'ls': '-', 'marker': 'v',
               'label': 'Exp Regret'},
}


def plot_results(results, gs, save_path='results.png', title_extra=''):
    fig, ax = plt.subplots(figsize=(8, 6))
    for method, data in results.items():
        if method not in STYLES or 'regrets' not in data: continue
        s = STYLES[method]
        arr = np.array(data['regrets'])
        if arr.size == 0: continue
        mean_v = np.mean(arr, axis=0); steps = np.arange(len(mean_v))
        if arr.shape[0] > 1:
            se = 1.645 * np.std(arr, axis=0) / np.sqrt(arr.shape[0])
            ax.fill_between(steps, np.maximum(mean_v-se, 1e-4),
                          mean_v+se, color=s['color'], alpha=0.15)
        ax.plot(steps, mean_v, color=s['color'], linestyle=s['ls'],
                marker=s['marker'], markevery=1, label=s['label'],
                linewidth=2, markersize=5)
    ax.set_yscale('log')
    ax.set_xlabel('Number of labeled samples after the warm-up period',
                  fontsize=11)
    ax.set_ylabel('log of the excess SPO risk', fontsize=11)
    title = f'Excess SPO Risk on a {gs}X{gs} Grid (Pool-Based)'
    if title_extra: title += f'\n{title_extra}'
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); print(f"Saved: {save_path}")


def plot_accuracy(results, gs, save_path='accuracy.png', title_extra=''):
    fig, ax = plt.subplots(figsize=(8, 6))
    for method, data in results.items():
        if method not in STYLES or 'accuracies' not in data: continue
        s = STYLES[method]
        arr = np.array(data['accuracies'])
        if arr.size == 0: continue
        mean_v = np.mean(arr, axis=0); steps = np.arange(len(mean_v))
        if arr.shape[0] > 1:
            se = 1.645 * np.std(arr, axis=0) / np.sqrt(arr.shape[0])
            ax.fill_between(steps, mean_v-se, mean_v+se,
                          color=s['color'], alpha=0.15)
        ax.plot(steps, mean_v, color=s['color'], linestyle=s['ls'],
                marker=s['marker'], markevery=1, label=s['label'],
                linewidth=2, markersize=5)
    ax.set_xlabel('Number of labeled samples after warm-up', fontsize=11)
    ax.set_ylabel('Path Accuracy', fontsize=11)
    title = f'{gs}x{gs} Shortest Path: Accuracy (Pool-Based)'
    if title_extra: title += f'\n{title_extra}'
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); print(f"Saved: {save_path}")


def plot_scan_counts(results, save_path='scans.png'):
    fig, ax = plt.subplots(figsize=(8, 5))
    for method, data in results.items():
        if method not in STYLES or 'scans' not in data: continue
        s = STYLES[method]
        ax.bar(s['label'], np.mean(data['scans']), yerr=np.std(data['scans']),
               color=s['color'], alpha=0.7, capsize=5)
    ax.set_ylabel('Total samples scanned', fontsize=11)
    ax.set_title('Samples scanned to reach budget', fontsize=13)
    ax.grid(True, alpha=0.3, axis='y'); plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); print(f"Saved: {save_path}")


# ============================================================
# Part 10: Core Experimental Function
# ============================================================

def run_experiment(
    mode='3x3',
    grid_size=None, p=None, noise_level=None, deg=None,
    n_init=None, n_pool=None, n_test=None, budget=None,
    margin_threshold=0.1,
    hidden_dim=None, model_type='linear',
    init_lr=0.01, init_epochs=10,
    update_lr=0.05, update_steps=3,
    online_mode=True,
    n_runs=None, M=None, methods=None,
    prior_precision=1.0, laplace_noise_var=1.0,
    stream_clustering=0.0,
    device='cpu', save_prefix='', verbose=True,
):

    presets = {
        'quick':  {'grid_size': 3, 'p': 5, 'n_init': 10, 'n_pool': 200,
                   'n_test': 200, 'budget': 15, 'noise_level': 0.25,
                   'deg': 1, 'hidden_dim': 64, 'n_runs': 3, 'M': 20},
        '3x3':   {'grid_size': 3, 'p': 5, 'n_init': 10, 'n_pool': 500,
                   'n_test': 1000, 'budget': 25, 'noise_level': 0.25,
                   'deg': 1, 'hidden_dim': 100, 'n_runs': 25, 'M': 30},
        '5x5':   {'grid_size': 5, 'p': 5, 'n_init': 10, 'n_pool': 1000,
                   'n_test': 1000, 'budget': 200, 'noise_level': 0.25,
                   'deg': 1, 'hidden_dim': 100, 'n_runs': 25, 'M': 30},
        'custom': {'grid_size': 3, 'p': 5, 'n_init': 10, 'n_pool': 500,
                   'n_test': 1000, 'budget': 25, 'noise_level': 0.25,
                   'deg': 1, 'hidden_dim': 100, 'n_runs': 10, 'M': 30},
    }
    assert mode in presets
    cfg = presets[mode].copy()
    for k, v in {'grid_size': grid_size, 'p': p, 'n_init': n_init,
                 'n_pool': n_pool, 'n_test': n_test, 'budget': budget,
                 'noise_level': noise_level, 'deg': deg,
                 'hidden_dim': hidden_dim, 'n_runs': n_runs, 'M': M}.items():
        if v is not None:
            cfg[k] = v

    gs = cfg['grid_size']
    d_edges, n_paths, all_paths = build_grid(gs)

    all_m = ['supervise', 'uncertainty', 'margin', 'flip', 'regret']
    methods_to_run = list(methods) if methods else all_m
    if 'supervise' not in methods_to_run:
        methods_to_run = ['supervise'] + methods_to_run

    if verbose:
        print("=" * 65)
        print(f"DFL Shortest Path - POOL-BASED")
        print("=" * 65)
        print(f"  Grid: {gs}x{gs}, d={d_edges}, {n_paths} paths")
        print(f"  Data: p={cfg['p']}, deg={cfg['deg']}, "
              f"noise={cfg['noise_level']}")
        print(f"  Split: init={cfg['n_init']}, pool={cfg['n_pool']}, "
              f"test={cfg['n_test']}, budget={cfg['budget']}")
        print(f"  Model: {model_type}, "
              f"training={'Online SGD (1 step)' if online_mode else 'warm-start'}, "
              f"lr={update_lr}")
        print(f"  Pool clustering: {stream_clustering:.2f}  "
              f"(0=random, 1=strictly clustered)")
        print(f"  Methods: {methods_to_run}")
        print(f"  Runs: {cfg['n_runs']}, M={cfg['M']}")
        print()

    results = {m: {'regrets': [], 'accuracies': [], 'times': [], 'scans': []}
               for m in methods_to_run}

    for run in range(cfg['n_runs']):
        if verbose:
            print(f"--- Run {run+1}/{cfg['n_runs']} ---")

        seed = run * 100 + 42
        rng_B = np.random.RandomState(seed)
        B, centers = generate_B_matrix(
            cfg['p'], d_edges, gs, all_paths,
            margin_threshold=margin_threshold, rng=rng_B)

        n_total = cfg['n_init'] + cfg['n_pool'] + cfg['n_test']
        X_all, C_all, _, _, ca_all = generate_data(
            n_total, cfg['p'], d_edges, gs, all_paths,
            noise_level=cfg['noise_level'], deg=cfg['deg'],
            seed=seed+1, B=B, centers=centers)

        X_init = X_all[:cfg['n_init']]
        C_init = C_all[:cfg['n_init']]
        X_pool = X_all[cfg['n_init']:cfg['n_init']+cfg['n_pool']]
        C_pool = C_all[cfg['n_init']:cfg['n_init']+cfg['n_pool']]
        ca_pool = ca_all[cfg['n_init']:cfg['n_init']+cfg['n_pool']]
        X_test = X_all[cfg['n_init']+cfg['n_pool']:]
        C_test = C_all[cfg['n_init']+cfg['n_pool']:]

        bayes = compute_bayes_risk(X_test, C_test, B, gs, cfg['p'],
                                    cfg['deg'])
        if verbose:
            print(f"  Bayes risk: {bayes:.4f}")

        rng_pool = np.random.RandomState(seed + 2)
        pool_order = make_pool_order(ca_pool, stream_clustering, rng_pool)

        for method in methods_to_run:
            if verbose:
                print(f"  Method: {method}", end='  ')

            np.random.seed(seed + 3)
            torch.manual_seed(seed)

            X_train = X_init.copy()
            C_train = C_init.copy()
            labeled_mask = np.zeros(cfg['n_pool'], dtype=bool)
            regrets = []
            accuracies = []
            times = []

            model = make_model(cfg['p'], cfg['hidden_dim'],
                               d_edges, model_type).to(device)
            train_init(model, X_train, C_train, gs, device,
                       epochs=init_epochs, lr=init_lr)

            m_eval = evaluate(model, X_test, C_test, gs, device)
            excess = max(m_eval['regret'] - bayes, 0.0)
            regrets.append(excess)
            accuracies.append(m_eval['path_accuracy'])

            if verbose:
                print(f"init excess={excess:.4f}, "
                      f"acc={m_eval['path_accuracy']:.3f}")

            t_start = time.time()

            if method == 'supervise':
                for step in range(cfg['budget']):
                    pool_idx = pool_order[step]
                    labeled_mask[pool_idx] = True
                    x_new = X_pool[pool_idx:pool_idx+1]
                    c_new = C_pool[pool_idx:pool_idx+1]
                    X_train = np.vstack([X_train, x_new])
                    C_train = np.vstack([C_train, c_new])

                    if online_mode:
                        train_online_step(model, x_new[0], c_new[0],
                                          gs, device, lr=update_lr)
                    else:
                        train_update(model, X_train, C_train, gs, device,
                                     steps=update_steps, lr=update_lr)

                    m_eval = evaluate(model, X_test, C_test, gs, device)
                    excess = max(m_eval['regret'] - bayes, 0.0)
                    regrets.append(excess)
                    accuracies.append(m_eval['path_accuracy'])
                    times.append(time.time() - t_start)

                    if verbose and (step+1) % max(1, cfg['budget']//5) == 0:
                        print(f"    [supervise] {step+1}/{cfg['budget']}: "
                              f"excess={excess:.4f}, "
                              f"acc={m_eval['path_accuracy']:.3f}")

                results[method]['regrets'].append(regrets)
                results[method]['accuracies'].append(accuracies)
                results[method]['times'].append(times)
                results[method]['scans'].append(cfg['budget'])
                continue

            laplace = None
            if method in ['flip', 'regret']:
                laplace = LastLayerLaplace(model, prior_precision,
                                           laplace_noise_var)
                laplace.fit(torch.FloatTensor(X_train).to(device), device)

            for step in range(cfg['budget']):
                unlabeled_idx = np.where(~labeled_mask)[0]

                # Refit Laplace before each scoring step
                if method in ['flip', 'regret']:
                    laplace = LastLayerLaplace(model, prior_precision,
                                               laplace_noise_var)
                    laplace.fit(torch.FloatTensor(X_train).to(device), device)

                scores = score_pool(
                    model, X_pool, X_train,
                    unlabeled_idx, method, gs, all_paths,
                    laplace=laplace, device=device, M=cfg['M'])

                best_local = int(np.argmax(scores))
                pool_idx = unlabeled_idx[best_local]
                labeled_mask[pool_idx] = True

                x_new = X_pool[pool_idx:pool_idx+1]
                c_new = C_pool[pool_idx:pool_idx+1]
                X_train = np.vstack([X_train, x_new])
                C_train = np.vstack([C_train, c_new])

                if online_mode:
                    train_online_step(model, x_new[0], c_new[0],
                                      gs, device, lr=update_lr)
                else:
                    train_update(model, X_train, C_train, gs, device,
                                 steps=update_steps, lr=update_lr)

                m_eval = evaluate(model, X_test, C_test, gs, device)
                excess = max(m_eval['regret'] - bayes, 0.0)
                regrets.append(excess)
                accuracies.append(m_eval['path_accuracy'])
                times.append(time.time() - t_start)

                if verbose and (step+1) % max(1, cfg['budget']//5) == 0:
                    print(f"    [{method}] {step+1}/{cfg['budget']}: "
                          f"excess={excess:.4f}, "
                          f"acc={m_eval['path_accuracy']:.3f}, "
                          f"pool[{pool_idx}] cluster={ca_pool[pool_idx]}")

            results[method]['regrets'].append(regrets)
            results[method]['accuracies'].append(accuracies)
            results[method]['times'].append(times)
            results[method]['scans'].append(cfg['budget'])

    tag = (f"{save_prefix}pool_{gs}x{gs}_deg{cfg['deg']}_"
           f"n{cfg['noise_level']}_clust{stream_clustering:.2f}")

    if verbose:
        print(f"\nGenerating plots...")

    plot_results(results, gs, f'results_{tag}.png')
    plot_accuracy(results, gs, f'accuracy_{tag}.png')
    plot_scan_counts(results, f'scans_{tag}.png')

    print(f"\n{'='*70}")
    print(f"Summary (Pool-Based): {gs}x{gs}, noise={cfg['noise_level']}, "
          f"clustering={stream_clustering:.2f}, {cfg['n_runs']} runs")
    print(f"{'='*70}")
    print(f"  {'Method':<18} {'Final Excess':>14} {'Final Acc':>12} "
          f"{'Avg Scans':>10}")
    print(f"  {'-'*58}")

    for method in methods_to_run:
        fr = [r[-1] for r in results[method]['regrets']]
        fa = [a[-1] for a in results[method]['accuracies']]
        fs = results[method]['scans']
        print(f"  {method:<18} {np.mean(fr):.4f}+/-{np.std(fr):.4f}  "
              f"{np.mean(fa):.3f}+/-{np.std(fa):.3f}  "
              f"{np.mean(fs):.0f}")

    if 'margin' in methods_to_run and 'supervise' in methods_to_run:
        print(f"\n  Mean excess per step:")
        header = f"  {'step':<6}"
        for m in methods_to_run:
            header += f" {m:<14}"
        print(header)

        arrays = {m: np.array(results[m]['regrets'])
                  for m in methods_to_run}
        n_steps = arrays['supervise'].shape[1]

        for i in range(0, n_steps, max(1, n_steps // 10)):
            row = f"  {i:<6}"
            for m in methods_to_run:
                row += f" {arrays[m][:, i].mean():<14.4f}"
            print(row)

    return results

In [ ]:
if __name__ == '__main__':
    results = run_experiment(
        mode='5x5',
        model_type='linear',
        #methods=['supervise', 'uncertainty'],
        n_runs=25,
        stream_clustering=1.0,
    )

DFL Shortest Path - POOL-BASED
  Grid: 5x5, d=40, 70 paths
  Data: p=5, deg=1, noise=0.25
  Split: init=10, pool=1000, test=1000, budget=200
  Model: linear, training=Online SGD (1 step), lr=0.05
  Pool clustering: 1.00  (0=random, 1=strictly clustered)
  Methods: ['supervise', 'uncertainty', 'margin', 'flip', 'regret']
  Runs: 25, M=30

--- Run 1/25 ---
  Bayes risk: 0.3421
  Method: supervise  init excess=2.6187, acc=0.014
    [supervise] 40/200: excess=1.2147, acc=0.075
    [supervise] 80/200: excess=1.0532, acc=0.127
    [supervise] 120/200: excess=0.6995, acc=0.179
    [supervise] 160/200: excess=0.5901, acc=0.194
    [supervise] 200/200: excess=0.3362, acc=0.297
  Method: uncertainty  init excess=2.6187, acc=0.014
    [uncertainty] 40/200: excess=0.4801, acc=0.223, pool[235] cluster=12
    [uncertainty] 80/200: excess=0.3241, acc=0.272, pool[59] cluster=41
    [uncertainty] 120/200: excess=0.4043, acc=0.242, pool[722] cluster=59
    [uncertainty] 160/200: excess=0.4122, acc=0.275